# Projeto de Predição de Falhas de Ativos com Cadeias de Markov
**Objetivo**: Prever eventos críticos (`Is_Dont_Go`) com uma antecedência de 4 horas utilizando sequências de alarmes de telemetria.

## 1. Configurações Iniciais e Monitoramento de Recursos

In [1]:
import os
import gc
import psutil
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from collections import deque
from tqdm import tqdm
from sklearn.metrics import (
    classification_report, 
    confusion_matrix, 
    precision_recall_curve, 
    accuracy_score, 
    precision_recall_fscore_support
)

def print_memory_status(etapa: str):
    """Centraliza o log de utilização de memória RAM e Swap."""
    mem = psutil.virtual_memory()
    swap = psutil.swap_memory()
    print(f"\n📈 [MEMÓRIA] - {etapa}")
    print(f"   RAM  -> Disponível: {mem.available / (1024**3):.2f} GB | Usada: {mem.used / (1024**3):.2f} GB ({mem.percent}%)")
    print(f"   SWAP -> Total: {swap.total / (1024**3):.2f} GB | Usado: {swap.used / (1024**3):.2f} GB ({swap.percent}%)")
    print("-" * 50)

print_memory_status("Carga inicial dos pacotes")


📈 [MEMÓRIA] - Carga inicial dos pacotes
   RAM  -> Disponível: 15.34 GB | Usada: 14.62 GB (48.8%)
   SWAP -> Total: 64.00 GB | Usado: 28.45 GB (44.5%)
--------------------------------------------------


## 2. Engenharia de Recursos e Modelagem Estatística

In [2]:
def extract_markov_transitions(df, window_hours=4, max_seq_len=3):
    """
    Percorre os dados em O(n) utilizando uma janela deslizante cronológica.
    Otimização O(1) na recuperação de estados para evitar travamentos por tempestades de alarmes.
    """
    df = df.sort_values(['TAG', 'Data_Evento']).reset_index(drop=True)
    
    transitions = []
    state_timeline = []
    current_tag = None
    window = deque()
    prev_state = None
    
    for row in tqdm(df.itertuples(), total=len(df), desc="⏳ Processando eventos"):
        tag = row.TAG
        t = row.Data_Evento
        alarm = row.Alarme
        is_dg = row.Is_Dont_Go
        
        if tag != current_tag:
            current_tag = tag
            window.clear()
            prev_state = None
            
        # Slide da janela temporal (Garante o horizonte de 4 horas)
        cutoff_time = t - pd.Timedelta(hours=window_hours)
        while window and window[0][0] < cutoff_time:
            window.popleft()
            
        # Construção do Novo Estado
        if is_dg == 1:
            new_state = "__DONT_GO__"
        else:
            window.append((t, alarm))
            
            # ACESSO O(1): Otimização essencial que impede o travamento do script
            w_len = len(window)
            take = min(w_len, max_seq_len)
            recent_alarms = [window[w_len - i][1] for i in range(take, 0, -1)]
            new_state = " -> ".join(recent_alarms)
            
        state_timeline.append(new_state)
        
        # Registro das transições S(t-1) -> S(t)
        if prev_state is not None:
            transitions.append({"from_state": prev_state, "to_state": new_state})
            
        # Controle de estados absorventes
        if is_dg == 1:
            prev_state = None
            window.clear()
        else:
            prev_state = new_state
            
    df_out = df.copy()
    df_out['state'] = state_timeline
    df_transitions = pd.DataFrame(transitions)
    
    return df_out, df_transitions


class MarkovChainPredictor:
    """Classe preditora baseada em matrizes de transição estocásticas."""
    ABSORBING_STATE = "__DONT_GO__"
    
    def __init__(self, n_steps=5):
        self.n_steps = n_steps
        
    def fit(self, transitions: pd.DataFrame):
        # Mapeamento do Espaço de Estados
        states_set = set(transitions["from_state"].unique()) | set(transitions["to_state"].unique())
        if self.ABSORBING_STATE not in states_set:
            states_set.add(self.ABSORBING_STATE)
            
        self.states_ = sorted(list(states_set))
        self.state_to_idx_ = {s: i for i, s in enumerate(self.states_)}
        self.n_states_ = len(self.states_)
        
        # Alocação O(N) da Matriz de Contagem
        self.count_matrix_ = np.zeros((self.n_states_, self.n_states_))
        from_idx = transitions["from_state"].map(self.state_to_idx_).values
        to_idx = transitions["to_state"].map(self.state_to_idx_).values
        np.add.at(self.count_matrix_, (from_idx, to_idx), 1)
        
        # Tratamento de ruídos no estado absorvente
        dg_idx = self.state_to_idx_[self.ABSORBING_STATE]
        self.count_matrix_[dg_idx, :] = 0
        
        # Laplace Smoothing
        self.count_matrix_ += 1e-6
        self.count_matrix_[dg_idx, :] = 0
        
        row_sums = self.count_matrix_.sum(axis=1, keepdims=True)
        row_sums[row_sums == 0] = 1 
        
        self.P_ = self.count_matrix_ / row_sums
        self.P_[dg_idx, dg_idx] = 1.0
        
        # Validações matemáticas obrigatórias
        self._validate_markov_chain()
        
        # Cálculo de risco prospectivo via exponenciação de matrizes
        P_n = np.linalg.matrix_power(self.P_, self.n_steps)
        self.risk_scores_ = {state: float(P_n[idx, dg_idx]) for state, idx in self.state_to_idx_.items()}
        return self

    def _validate_markov_chain(self):
        print("\n" + "="*50)
        print("📐 RELATÓRIO DE VALIDAÇÃO DA CADEIA DE MARKOV\n" + "="*50)
        row_sums = self.P_.sum(axis=1)
        print(f"✔ Matriz Estocástica P(linha)=1: {np.allclose(row_sums, 1.0)}")
        col_sums = self.count_matrix_.sum(axis=0)
        print(f"✔ Estados inalcançáveis: {len([x for x in col_sums if x <= 1e-5])}")
        
    def predict_proba(self, states: pd.Series) -> np.ndarray:
        known_values = list(self.risk_scores_.values())
        default = float(np.median(known_values)) if known_values else 0.0
        return np.array([self.risk_scores_.get(s, default) for s in states])

## 3. Pipeline de Treino e Validação (Mês de Janeiro)

In [ ]:
# Configuração de Pipeline
mes_escolhido = "jan"
caminho_entrada = os.path.join("..", "data", "telemetry", "processed", f"{mes_escolhido}.csv")

print(f"⏳ Carregando base histórica de desenvolvimento: {caminho_entrada}...")
df_raw = pd.read_csv(caminho_entrada, low_memory=False)
df_raw['Data_Evento'] = pd.to_datetime(df_raw['Data_Evento'])
df_raw = df_raw.sort_values(['TAG', 'Data_Evento']).reset_index(drop=True)

# Definição do Alvo Preditivo (Janela de Antecedência de 4h)
falhas = df_raw[df_raw['Is_Dont_Go'] == 1][['TAG', 'Data_Evento']].rename(columns={'Data_Evento': 'Data_Falha'})
df_raw = df_raw.sort_values('Data_Evento')
falhas = falhas.sort_values('Data_Falha')

df_raw = pd.merge_asof(df_raw, falhas, by='TAG', left_on='Data_Evento', right_on='Data_Falha', direction='forward')
df_raw = df_raw.sort_values(['TAG', 'Data_Evento']).reset_index(drop=True)
df_raw['TTF'] = df_raw['Data_Falha'] - df_raw['Data_Evento']
df_raw['Target_4h'] = ((df_raw['TTF'] > pd.Timedelta(seconds=0)) & (df_raw['TTF'] <= pd.Timedelta(hours=4))).astype(int)

# Processamento de Estados
df_states, _ = extract_markov_transitions(df_raw, window_hours=4, max_seq_len=3)

# Divisão Cronológica de Validação (Holdout 70/30)
split_date = df_states["Data_Evento"].quantile(0.70)
df_train = df_states[df_states["Data_Evento"] <= split_date].copy()
df_test = df_states[df_states["Data_Evento"] > split_date].copy()

# Extração de transições e ajuste do modelo
_, train_transitions = extract_markov_transitions(df_train, window_hours=4, max_seq_len=3)
mc_model = MarkovChainPredictor(n_steps=20)
mc_model.fit(train_transitions)

# Escoragem do conjunto de teste
df_test["risk_score"] = mc_model.predict_proba(df_test['state']).astype(float)
print_memory_status("Fim do Treinamento")

⏳ Carregando base histórica de desenvolvimento: ../data/telemetry/processed/jan.csv...


⏳ Processando eventos: 100%|██████████| 3780001/3780001 [00:26<00:00, 142547.54it/s]



📐 RELATÓRIO DE VALIDAÇÃO DA CADEIA DE MARKOV
✔ Matriz Estocástica P(linha)=1: True
✔ Estados inalcançáveis: 0


## 4. Análise de Resultados e Calibração de Limiar

In [ ]:
print("--- Otimização do Limiar (Threshold) para Janela Preditiva de 4h ---")

# Avalia performance apenas em janelas estáveis anteriores ao evento consumado
df_test_eval = df_test[df_test['Is_Dont_Go'] == 0].copy()
y_pred_prob = df_test_eval['risk_score'].values
y_test = df_test_eval['Target_4h'].fillna(0).astype(int).values 

# Calibração do Threshold via curva Precision-Recall
precisions, recalls, thresholds = precision_recall_curve(y_test, y_pred_prob)
f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-10)
best_idx = np.argmax(f1_scores)
best_threshold = thresholds[best_idx]

print(f"Melhor Threshold Calibrado: {best_threshold:.4f}")
print(f"F1-Score: {f1_scores[best_idx]:.4f} | Recall: {recalls[best_idx]:.4f} | Precisão: {precisions[best_idx]:.4f}\n")

y_pred = (y_pred_prob >= best_threshold).astype(int)
print(classification_report(y_test, y_pred, target_names=['Classe 0 (Estável)', 'Classe 1 (Alerta 4h)']))

# Geração dos Artefatos Corporativos
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False)
plt.title(f'Matriz de Confusão Preditiva - {mes_escolhido.upper()}')

pasta_resultados = os.path.join('..', 'resultados', mes_escolhido)
os.makedirs(pasta_resultados, exist_ok=True)
plt.savefig(os.path.join(pasta_resultados, f'{mes_escolhido}_matriz_confusao.png'), bbox_inches='tight')
plt.show()

## 5. Extrapolação do Modelo para Outros Períodos (Rollout Temporal)

In [ ]:
meses_restantes = ["feb", "mar", "abr", "may", "jun"]

for mes in meses_restantes:
    print("\n" + "="*60)
    print(f"🔮 EXECUTANDO ROLLOUT TEMPORAL OTIMIZADO: {mes.upper()}")
    print("="*60)
    
    caminho_entrada = os.path.join("..", "data", "telemetry", "processed", f"{mes}.csv")
    if not os.path.exists(caminho_entrada): continue
        
    try:
        df_raw = pd.read_csv(caminho_entrada, low_memory=False)
        df_raw['Data_Evento'] = pd.to_datetime(df_raw['Data_Evento'])
        df_raw = df_raw.sort_values(['TAG', 'Data_Evento']).reset_index(drop=True)
        
        falhas = df_raw[df_raw['Is_Dont_Go'] == 1][['TAG', 'Data_Evento']].rename(columns={'Data_Evento': 'Data_Falha'})
        df_raw = df_raw.sort_values('Data_Evento')
        falhas = falhas.sort_values('Data_Falha')
        
        df_raw = pd.merge_asof(df_raw, falhas, by='TAG', left_on='Data_Evento', right_on='Data_Falha', direction='forward')
        df_raw = df_raw.sort_values(['TAG', 'Data_Evento']).reset_index(drop=True)
        df_raw['TTF'] = df_raw['Data_Falha'] - df_raw['Data_Evento']
        df_raw['Target_4h'] = ((df_raw['TTF'] > pd.Timedelta(seconds=0)) & (df_raw['TTF'] <= pd.Timedelta(hours=4))).astype(int)
        
        # Chamada à função com otimização O(1) - Processa Junho em segundos
        df_states, _ = extract_markov_transitions(df_raw, window_hours=4, max_seq_len=3)
        df_states["risk_score"] = mc_model.predict_proba(df_states['state']).astype(float)
        
        df_eval = df_states[df_states['Is_Dont_Go'] == 0].copy()
        y_pred_prob = df_eval['risk_score'].values
        y_test = df_eval['Target_4h'].fillna(0).astype(int).values
        
        precisions, recalls, thresholds = precision_recall_curve(y_test, y_pred_prob)
        f1_scores = 2 * (precisions * recalls) / (precisions * recalls + 1e-10)
        best_threshold = thresholds[np.argmax(f1_scores)]
        y_pred = (y_pred_prob >= best_threshold).astype(int)
        
        # Salvamento de Métricas em disco
        pasta_res = os.path.join('..', 'resultados', mes)
        os.makedirs(pasta_res, exist_ok=True)
        
        acc = accuracy_score(y_test, y_pred)
        precision, recall, f1, _ = precision_recall_fscore_support(y_test, y_pred, average=None)
        pd.DataFrame({
            'Metrica': ['Acuracia', 'Precision_1', 'Recall_1', 'F1_1'],
            'Valor': [acc, precision[1], recall[1], f1[1]]
        }).to_csv(os.path.join(pasta_res, f'{mes}_metricas.csv'), index=False)
        
        print(f"✨ Avaliação de {mes.upper()} concluída com sucesso!")
        
    except Exception as e:
        print(f"❌ Erro no mês {mes.upper()}: {str(e)}")
    finally:
        try: del df_raw, df_states, df_eval
        except NameError: pass
        gc.collect()

print("\n🏁 [FIM]: Pipeline executado sem gargalos de processamento.")